# CRYSTALS-Kyber with Complete Mathematical Walkthrough

**Post-Quantum Key Encapsulation Mechanism | ML-KEM / FIPS 203**

This notebook teaches the **complete mathematics** of the Kyber crystal algorithm.
Every concept is explained first, then immediately implemented in Python.

**Toy parameters** with the same structure as FIPS 203, just smaller numbers:

| Symbol | Value | Meaning |
|--------|-------|---------|
| q | 17 | Prime modulus, all coefficients live in {0 … 16} |
| n | 8 | Polynomial degree, ring modulus is X⁸ + 1 |
| k | 2 | Module dimension, A is 2×2, vectors have 2 polynomials |
| η | 2 | CBD noise bound, secret coefficients in {−2,−1,0,1,2} |

**How to use this notebook:**
1. Run **Module 0: Global Setup** first. It defines all shared constants.
2. Work through each module in order. Every cell builds on the previous.
3. All arithmetic is mod **q = 17** unless stated otherwise.

> **All numbers are self-consistent and verified by the assertions in each cell.** The ring, the negacyclic NTT (via psi-twisting), and the KeyGen->Encaps->Decaps flow mirror the structure of FIPS 203 at toy scale.

## Module 0: The Global Setup

> **Run this cell first.** Every other cell depends on these constants.
> Do not modify the values below. They are the verified ground truth for all exercises.

The constants are split into three groups:
- **Parameters** which are the mathematical settings of this Kyber instance
- **Key Generation values:** secret key s, error e, NTT roots, public key t
- **Encapsulation and Decapsulation values:** ciphertext u, v and recovery w

In [ ]:
# Setup: toy parameters, NTT twist tables, and reference values (run this cell first)

# Toy parameters:
Q     = 17   # prime modulus
N     = 8    # polynomial degree, ring is X^8 + 1 (negacyclic)
K     = 2    # module dimension
ETA   = 2    # CBD noise parameter eta
N_INV = 15   # n^-1: 8 x 15 = 120 = 1 mod 17

# NTT roots (omega = 9 is a primitive n-th = 8th root of unity)
ROOTS     = [1, 9, 13, 15, 16, 8, 4, 2]
INV_ROOTS = [1, 2,  4,  8, 16, 15, 13, 9]

# Negacyclic twist tables (psi = 3 is a primitive 2n-th = 16th root, psi^8 = -1).
# These turn the plain (cyclic) NTT into multiplication in Z_17[X]/(X^8 + 1).
PSI         = 3
PSI_INV     = 6                                  # 3 x 6 = 18 = 1 mod 17
PSI_POW     = [1, 3, 9, 10, 13, 5, 15, 11]      # psi^i
PSI_INV_POW = [1, 6, 2, 12, 4, 7, 8, 14]  # psi^-i

# Matrix A (NTT domain, generated from public seed rho)
A = [
    [[2,11,8,1,15,3,3,6],  [6,6,9,9,12,12,15,15]],
    [[7,14,3,10,5,12,1,8], [3,7,2,9,14,4,11,6  ]]
]

# Secret key s and error e (CBD output, normal domain)
S0 = [1, 0, 0, 1, 0, 0, 0, 1]
S1 = [0, 1, 0, 0, 1, 0, 0, 0]
E0 = [1, 0, 0, 0, 0, 0, 0, 0]
E1 = [0, 0, 1, 0, 0, 0, 0, 0]

# NTT domain forms of s and e (negacyclic NTT)
S0_NTT = [5, 3, 0, 9, 14, 16, 2, 10]
S1_NTT = [16, 14, 1, 15, 10, 11, 8, 10]
E0_NTT = [1, 1, 1, 1, 1, 1, 1, 1]
E1_NTT = [9, 15, 8, 2, 9, 15, 8, 2]

# Public key t (NTT domain)
T0 = [5, 16, 10, 9, 8, 11, 8, 7]
T1 = [7, 2, 10, 6, 15, 13, 13, 6]

# Encapsulation: fresh noise and message
R0    = [1, 0, 1, 0, 0, 1, 0, 0]
R1    = [0, 1, 0, 0, 1, 0, 1, 0]
E1_0  = [0, 0, 0, 1, 0, 0, 0, 0]
E1_1  = [0, 0, 0, 0, 0, 1, 0, 0]
E2    = [0, 0, 0, 0, 1, 0, 0, 0]
R0_NTT = [15, 5, 6, 13, 5, 10, 12, 10]
R1_NTT = [14, 6, 3, 6, 8, 3, 10, 1]
MSG   = [1, 0, 1, 0, 1, 0, 1, 0]
M_HAT = [8, 0, 8, 0, 8, 0, 8, 0]   # message encoded: bit 1 -> 8, bit 0 -> 0

# Ciphertext
U0 = [10, 8, 14, 4, 0, 4, 3, 7]
U1 = [10, 11, 10, 16, 2, 11, 4, 7]
V  = [14, 8, 3, 0, 5, 12, 11, 3]

# Decapsulation intermediate values
U0_NTT = [2, 0, 12, 10, 3, 1, 6, 12]
U1_NTT = [1, 10, 6, 11, 14, 7, 4, 10]
STU_N  = [6, 7, 10, 0, 13, 11, 4, 3]
W      = [8, 1, 10, 0, 9, 1, 7, 0]
DECODED= [1, 0, 1, 0, 1, 0, 1, 0]

print(f"Setup complete.  Q={Q}  N={N}  K={K}  ETA={ETA}  N_INV={N_INV}")
print(f"omega roots:  {ROOTS}")
print(f"psi^i twist:  {PSI_POW}")


## Module 1: The Mathematical Foundations

### 1.1 The Ring R_q = Z₁₇[X] / (X⁸ + 1)

All Kyber arithmetic lives in this ring:
- **Every polynomial has at most 8 terms** in other words coefficients on X⁰ through X⁷
- **All coefficients are integers mod q = 17** and they live in {0, 1, …, 16}
- **Negacyclic reduction:** when multiplication produces X^k with k ≥ 8, we use the identity
 `X⁸ ≡ −1 mod (X⁸+1)`, giving:
 `X⁸ → −1`, `X⁹ → −X`, `X¹⁰ → −X²`, …, `X¹⁵ → −X⁷`

**Example:** a coefficient on X¹⁰ wraps as `−1 × (coefficient on X²) mod 17`

### 1.2 Finding the primitive root g

We need a generator g of Z*₁₇ is an element whose powers produce all 16 nonzero residues mod 17.

**Test:** g must have **order 16**, meaning g^16 ≡ 1 mod 17 AND g^k ≠ 1 for all 1 ≤ k < 16.
Equivalently (since 16 = 2⁴, only prime factor is 2): check that g^8 ≠ 1 mod 17.

- g = 2 **fails**: 2⁸ mod 17 = 1 → order is only 8, not 16
- g = 3 **works**: 3⁸ mod 17 = 16 = −1 ≠ 1 → order is 16 ✓

### 1.3 Deriving ψ, ω, and the roots tables

From g = 3 we derive everything needed for the NTT:

```
ψ = g^((q−1)/(2n)) mod q = 3^((17−1)/(2×8)) = 3¹ mod 17 = 3
 Check: ψ^16 = 1 mod 17 ✓ (primitive 2n-th root of unity)
 Check: ψ^8 = 16 = −1 mod 17 ✓ (must not equal 1)

ω = ψ² mod 17 = 9
 Check: ω^8 = 1 mod 17 ✓ (primitive n-th root of unity)

ω⁻¹ = 2 because 9 × 2 = 18 ≡ 1 mod 17
n⁻¹ = 15 because 8 × 15 = 120 ≡ 1 mod 17

roots[k] = ω^k mod 17 = [1, 9, 13, 15, 16, 8, 4, 2]
inv_roots[k] = (ω⁻¹)^k mod 17 = [1, 2, 4, 8, 16, 15, 13, 9]
```

**Verify:** `roots[k] × inv_roots[k] ≡ 1 mod 17` for all k

### Exercise 1: Implement `modpow` and verify g, ψ, ω

Implement fast modular exponentiation using the **square-and-multiply** method:

```
result = 1
while exp > 0:
 if exp is odd: result = result × base mod mod
 base = base² mod mod
 exp = exp // 2
```

Then use it to verify the derivation chain g → ψ → ω → roots tables.

In [ ]:
def modpow(base: int, exp: int, mod: int) -> int:
    """Fast modular exponentiation (square-and-multiply)."""
    result, b = 1, base % mod
    while exp > 0:
        if exp & 1:            # if exp is odd
            result = result * b % mod
        b = b * b % mod        # square the base
        exp >>= 1              # exp = exp // 2
    return result

# Verify g = 2 fails
print("Does g = 2 work?")
print(f"2^8  mod 17 = {modpow(2,  8, Q)}   (need ≠ 1 for full order 16)")
print(f"2^16 mod 17 = {modpow(2, 16, Q)}   (order is 8, not 16 → REJECTED ✗)")

# Verify g = 3 works
print("\nDoes g = 3 work?")
print("k    3^k mod 17")
elements = set()
for k in range(1, 17):
    v = modpow(3, k, Q)
    elements.add(v)
    mark = "  ← identity, order = 16 confirmed ✓" if v == 1 else ""
    print(f"  {k:2d}     {v:2d}{mark}")
print(f"\nGenerated {len(elements)} distinct values → g = 3 ACCEPTED ✓")

# Derive ψ, ω, inverses
psi       = modpow(3, (Q-1)//(2*N), Q)   # ψ = 3^1 mod 17 = 3
omega     = modpow(psi, 2, Q)             # ω = ψ² = 9
omega_inv = modpow(omega, Q-2, Q)         # ω⁻¹ via Fermat: ω^(q-2)
n_inv_v   = modpow(N, Q-2, Q)            # n⁻¹ = 8^15 mod 17 = 15

print(f"\nψ  = g^((q-1)/(2n)) = {psi}")
print(f"   ψ^16 = {modpow(psi, 16, Q)}  (must be 1) ✓")
print(f"   ψ^8  = {modpow(psi,  8, Q)}  (must be 16 = −1) ✓")
print(f"ω  = ψ² mod 17 = {omega}")
print(f"   ω^8  = {modpow(omega, 8, Q)}  (must be 1) ✓")
print(f"ω⁻¹ = {omega_inv}  check: {omega} × {omega_inv} mod 17 = {omega*omega_inv%Q}")
print(f"n⁻¹ = {n_inv_v}  check: {N} × {n_inv_v} mod 17 = {N*n_inv_v%Q}")

# Build and verify roots tables
fwd = [modpow(omega,     k, Q) for k in range(N)]
inv = [modpow(omega_inv, k, Q) for k in range(N)]
print(f"\nForward roots:  {fwd}")
print(f"Inverse roots:  {inv}")
check = [fwd[k] * inv[k] % Q for k in range(N)]
print(f"Product check:  {check}  (all must be 1)")

assert fwd == ROOTS,     "Forward roots mismatch!"
assert inv == INV_ROOTS, "Inverse roots mismatch!"
print("\n✓ Both roots tables match the global constants.")

## Module 2 CBD Sampling: Generating s and e

### 2.1 Why small coefficients?

Kyber's security depends on the secret key **s** and error **e** having **small** coefficients.
The noise term in decapsulation must satisfy `|noise| < q/4 = 4.25` for correct decryption.
CBD with η=2 guarantees coefficients in {−2, −1, 0, +1, +2}, which is exactly small enough.

### 2.2 The CBD algorithm (η = 2)

Input: a byte stream from `SHAKE256(σ ‖ counter)`, read **LSB-first** (bit 0 of byte first).

For every 4 bits `(b₀, b₁, b₂, b₃)`:
```
coefficient = (b₀ + b₁) − (b₂ + b₃)
```
This has the **centered binomial distribution** with parameter η=2:
values −2, −1, 0, +1, +2 with probabilities 1/16, 4/16, 6/16, 4/16, 1/16.

**Negative coefficients** are stored as their positive representative mod q:
`−1 → 16`, `−2 → 15`

**One byte → 2 coefficients. Four bytes → one polynomial (n=8 coefficients).**

### 2.3 Full trace from the reference document

```
s[0] input bytes: [0x01, 0x10, 0x00, 0x10]

Byte 0x01 = 00000001 (LSB-first: 1,0,0,0,0,0,0,0)
 coeff[0]: (b0+b1)−(b2+b3) = (1+0)−(0+0) = +1 → stored: 1
 coeff[1]: (b4+b5)−(b6+b7) = (0+0)−(0+0) = 0 → stored: 0

Byte 0x10 = 00010000 (LSB-first: 0,0,0,0,1,0,0,0)
 coeff[2]: (0+0)−(0+0) = 0 → stored: 0
 coeff[3]: (1+0)−(0+0) = +1 → stored: 1

Byte 0x00 → coeff[4]=0, coeff[5]=0
Byte 0x10 → coeff[6]=0, coeff[7]=+1

s[0] = [1, 0, 0, 1, 0, 0, 0, 1] ✓
```

### Exercise 2: Implement `cbd_sample`

Convert a list of bytes to polynomial coefficients using the CBD algorithm.

In [ ]:
def cbd_sample(byte_list: list, q: int) -> list:
    """
    CBD η=2: convert bytes to polynomial coefficients.
    Each byte → 2 coefficients (4 bits each, LSB-first).
    Returns len(byte_list) × 2 coefficients, each in {0 … q-1}.
    """
    coefficients = []
    for byte_val in byte_list:
        # Extract all 8 bits, LSB first
        bits = [(byte_val >> i) & 1 for i in range(8)]
        # Two groups of 4 bits per byte
        for pair in range(2):
            b0, b1, b2, b3 = bits[pair*4 : pair*4+4]
            coeff = (b0 + b1) - (b2 + b3)   # centered binomial
            coefficients.append(coeff % q)  # store positive representative
    return coefficients

print("CBD for s[0] bytes [0x01, 0x10, 0x00, 0x10]\n")
s0_bytes = [0x01, 0x10, 0x00, 0x10]
for idx, byte_val in enumerate(s0_bytes):
    bits = [(byte_val >> i) & 1 for i in range(8)]
    print(f"  Byte 0x{byte_val:02X} = {byte_val:08b}  (LSB-first: {bits})")
    for pair in range(2):
        b0, b1, b2, b3 = bits[pair*4 : pair*4+4]
        raw    = (b0+b1) - (b2+b3)
        stored = raw % Q
        print(f"    coeff[{idx*2+pair}]: ({b0}+{b1})−({b2}+{b3}) = {raw:+d}  →  stored: {stored}")
    print()

s0 = cbd_sample([0x01, 0x10, 0x00, 0x10], Q)
s1 = cbd_sample([0x10, 0x00, 0x01, 0x00], Q)
e0 = cbd_sample([0x01, 0x00, 0x00, 0x00], Q)
e1 = cbd_sample([0x00, 0x01, 0x00, 0x00], Q)

print(f"s[0] = {s0}  expected: {S0}")
print(f"s[1] = {s1}  expected: {S1}")
print(f"e[0] = {e0}  expected: {E0}")
print(f"e[1] = {e1}  expected: {E1}")

assert s0==S0 and s1==S1 and e0==E0 and e1==E1
print("\n✓ All CBD outputs match the global constants.")

# Edge cases
print("\nEdge cases:")
print(f"  0xFF → {cbd_sample([0xFF], Q)}  (1111|1111 → (1+1)−(1+1)=0 twice)")
print(f"  0x0F → {cbd_sample([0x0F], Q)}  (1111|0000 → +2 then 0)")
print(f"  0xF0 → {cbd_sample([0xF0], Q)}  (0000|1111 → 0 then −2=15)")

## Module 3: Forward NTT (Cooley-Tukey) and the Negacyclic Twist

### 3.1 Why NTT?

Polynomial multiplication in R_q naively takes **O(n2)** operations.
The **Number Theoretic Transform** moves polynomials to a domain where
multiplication is **pointwise**, reducing the cost to **O(n log n)**.

### 3.2 Cyclic vs negacyclic: why we need psi

A plain Cooley-Tukey NTT built from `omega` (a primitive n-th root, here omega = 9)
computes the **cyclic** convolution, i.e. multiplication mod `X^n - 1`.
But Kyber's ring is **negacyclic**: `Z_q[X]/(X^n + 1)`, where `X^n = -1`.

To bridge the gap we use `psi`, a primitive **2n-th** root with `psi^n = -1` (here psi = 3):

```
forward:  a_hat   = NTT_omega( psi^i  * a[i] )     # pre-twist, then cyclic NTT
inverse:  a[i]    = psi^-i * INTT_omega( a_hat )   # cyclic INTT, then post-twist
```

The pre-twist by `psi^i` and post-twist by `psi^-i` are exactly what turn the cyclic
transform into multiplication in `X^n + 1`. With psi = 3: `psi^i = [1,3,9,10,13,5,15,11]`.

### 3.3 How real Kyber differs (fidelity note)

Real Kyber (q = 3329, n = 256) has **no primitive 512th root of unity**, so its NTT is
**incomplete**: 7 layers that stop at 128 degree-1 pairs, finished by a `basemul` step.
Our toy (q = 17, n = 8) **does** have a primitive 16th root (psi = 3), so we can run a
**complete** NTT all the way down to points. Same idea, different depth: don't assume real
Kyber's NTT reaches single points; it does not.

### 3.4 Bit-reversal permutation

Before the butterfly layers the array is permuted by reversing the binary index.
For n = 8: pairs (1,4) and (3,6) swap; (0, 2, 5, 7) stay.

### 3.5 Butterfly operation

```
t = v * w mod q ; new_u = (u + t) mod q ; new_v = (u - t) mod q
```
with three layers (len = 2, 4, 8), `step = n // len`, twiddle `w = roots[j * step]`.

### 3.6 Full negacyclic trace for s[0] = [1,0,0,1,0,0,0,1]

```
Pre-twist by psi^i (psi=3, psi^i = [1, 3, 9, 10, 13, 5, 15, 11]):
  s[0]        = [1, 0, 0, 1, 0, 0, 0, 1]
  s[0]*psi^i  = [1, 0, 0, 10, 0, 0, 0, 11]
After bit-reversal: [1, 0, 0, 0, 0, 0, 10, 11]
Layer 1 (len=2, step=4):
  [0,1] u= 1 v= 0 w= 1 -> t= 0  new[0]= 1 new[1]= 1
  [2,3] u= 0 v= 0 w= 1 -> t= 0  new[2]= 0 new[3]= 0
  [4,5] u= 0 v= 0 w= 1 -> t= 0  new[4]= 0 new[5]= 0
  [6,7] u=10 v=11 w= 1 -> t=11  new[6]= 4 new[7]=16
  After layer 1: [1, 1, 0, 0, 0, 0, 4, 16]
Layer 2 (len=4, step=2):
  [0,2] u= 1 v= 0 w= 1 -> t= 0  new[0]= 1 new[2]= 1
  [1,3] u= 1 v= 0 w=13 -> t= 0  new[1]= 1 new[3]= 1
  [4,6] u= 0 v= 4 w= 1 -> t= 4  new[4]= 4 new[6]=13
  [5,7] u= 0 v=16 w=13 -> t= 4  new[5]= 4 new[7]=13
  After layer 2: [1, 1, 1, 1, 4, 4, 13, 13]
Layer 3 (len=8, step=1):
  [0,4] u= 1 v= 4 w= 1 -> t= 4  new[0]= 5 new[4]=14
  [1,5] u= 1 v= 4 w= 9 -> t= 2  new[1]= 3 new[5]=16
  [2,6] u= 1 v=13 w=13 -> t=16  new[2]= 0 new[6]= 2
  [3,7] u= 1 v=13 w=15 -> t= 8  new[3]= 9 new[7]=10
  After layer 3: [5, 3, 0, 9, 14, 16, 2, 10]
NTT(s[0]) = [5, 3, 0, 9, 14, 16, 2, 10]
```

### Exercise 3: Implement `ntt` (pre-twist + cyclic Cooley-Tukey)


In [ ]:
import math

def bit_reverse_permute(a: list) -> list:
    """Reorder elements by bit-reversed indices."""
    n    = len(a)
    bits = int(math.log2(n))
    r    = list(a)
    for i in range(n):
        rev = int(f'{i:0{bits}b}'[::-1], 2)
        if rev > i:
            r[i], r[rev] = r[rev], r[i]
    return r

def _cyclic(a: list, roots: list, q: int) -> list:
    """Plain Cooley-Tukey NTT. On its own this realizes CYCLIC convolution
    (mod X^n - 1). The psi-twist in ntt()/intt() upgrades it to the
    negacyclic ring mod X^n + 1 that Kyber actually uses."""
    n = len(a)
    a = bit_reverse_permute(a)
    length = 2
    while length <= n:
        step = n // length
        for bs in range(0, n, length):
            for j in range(length // 2):
                f, s = bs + j, bs + j + length // 2
                w    = roots[j * step]
                t    = a[s] * w % q
                a[f], a[s] = (a[f] + t) % q, (a[f] - t) % q
        length <<= 1
    return a

def ntt(poly: list, roots: list, q: int) -> list:
    """Negacyclic forward NTT: pre-twist by psi^i, then the cyclic transform."""
    n = len(poly)
    twisted = [poly[i] * PSI_POW[i] % q for i in range(n)]
    return _cyclic(twisted, roots, q)

# Confirm against the global constants
print("s[0]              =", S0)
print("s[0] * psi^i      =", [S0[i]*PSI_POW[i] % Q for i in range(N)])
print("NTT(s[0])         =", ntt(S0, ROOTS, Q), " expected", S0_NTT)
assert ntt(S0, ROOTS, Q) == S0_NTT
assert ntt(S1, ROOTS, Q) == S1_NTT
assert ntt(E0, ROOTS, Q) == E0_NTT
assert ntt(E1, ROOTS, Q) == E1_NTT
print("\nAll NTT results match the global constants.")


## Module 4: Inverse NTT

### 4.1 Structure

The INTT reuses the **same cyclic butterfly** as the forward transform, with three steps:

1. Run the cyclic transform with `inv_roots` instead of `roots`.
2. Multiply every coefficient by `n^-1 = 15`.
3. **Post-twist** by `psi^-i` (psi^-1 = 6) to undo the forward `psi^i` twist.

Steps 2-3 are what make this a true negacyclic inverse, so that
`intt(ntt(a)) == a` in `Z_17[X]/(X^8 + 1)`.

### 4.2 Round-trip trace for NTT(s[0])

```
Input  NTT(s[0]) = [5, 3, 0, 9, 14, 16, 2, 10]
After bit-reversal: [5, 14, 0, 2, 3, 16, 9, 10]
Layer 1 (len=2, step=4):
  [0,1] u= 5 v=14 w= 1 -> t=14  new[0]= 2 new[1]= 8
  [2,3] u= 0 v= 2 w= 1 -> t= 2  new[2]= 2 new[3]=15
  [4,5] u= 3 v=16 w= 1 -> t=16  new[4]= 2 new[5]= 4
  [6,7] u= 9 v=10 w= 1 -> t=10  new[6]= 2 new[7]=16
  After layer 1: [2, 8, 2, 15, 2, 4, 2, 16]
Layer 2 (len=4, step=2):
  [0,2] u= 2 v= 2 w= 1 -> t= 2  new[0]= 4 new[2]= 0
  [1,3] u= 8 v=15 w= 4 -> t= 9  new[1]= 0 new[3]=16
  [4,6] u= 2 v= 2 w= 1 -> t= 2  new[4]= 4 new[6]= 0
  [5,7] u= 4 v=16 w= 4 -> t=13  new[5]= 0 new[7]= 8
  After layer 2: [4, 0, 0, 16, 4, 0, 0, 8]
Layer 3 (len=8, step=1):
  [0,4] u= 4 v= 4 w= 1 -> t= 4  new[0]= 8 new[4]= 0
  [1,5] u= 0 v= 0 w= 2 -> t= 0  new[1]= 0 new[5]= 0
  [2,6] u= 0 v= 0 w= 4 -> t= 0  new[2]= 0 new[6]= 0
  [3,7] u=16 v= 8 w= 8 -> t=13  new[3]=12 new[7]= 3
  After layer 3: [8, 0, 0, 12, 0, 0, 0, 3]
Before scaling: [8, 0, 0, 12, 0, 0, 0, 3]
Multiply by n^-1 = 15: [1, 0, 0, 10, 0, 0, 0, 11]
Post-twist by psi^-i (psi^-1=6): [1, 0, 0, 1, 0, 0, 0, 1]
```

### Exercise 4: Implement `intt`


In [ ]:
def intt(poly: list, inv_roots: list, q: int, n_inv: int) -> list:
    """Negacyclic inverse NTT:
        1. cyclic transform with inv_roots
        2. scale every coefficient by n^-1
        3. post-twist by psi^-i  (undoes the forward psi^i twist)."""
    n      = len(poly)
    a      = _cyclic(poly, inv_roots, q)
    scaled = [x * n_inv % q for x in a]
    return [scaled[i] * PSI_INV_POW[i] % q for i in range(n)]

print("INTT(NTT(s[0])) =", intt(S0_NTT, INV_ROOTS, Q, N_INV), " expected", S0)
assert intt(S0_NTT, INV_ROOTS, Q, N_INV) == S0
assert intt(S1_NTT, INV_ROOTS, Q, N_INV) == S1
assert intt(E0_NTT, INV_ROOTS, Q, N_INV) == E0
assert intt(E1_NTT, INV_ROOTS, Q, N_INV) == E1
print("All INTT round-trips match the global constants.")


## Module 5: Public Key  t_hat = A_hat . s_hat + e_hat

### 5.1 Everything stays in the NTT domain

Once s, e are transformed (s_hat, e_hat) and A is in NTT domain (A_hat),
the public key needs only **pointwise** operations, so no INTT here:

```
t_hat[0] = A_hat[0][0] (x) s_hat[0] + A_hat[0][1] (x) s_hat[1] + e_hat[0]
t_hat[1] = A_hat[1][0] (x) s_hat[0] + A_hat[1][1] (x) s_hat[1] + e_hat[1]
```

`(x)` = pointwise multiply mod q, `+` = pointwise add mod q.

### 5.2 Worked column for t_hat[0], coefficient i = 0

With the negacyclic s_hat[0] = [5, 3, 0, 9, 14, 16, 2, 10]:

```
A_hat[0][0][0] = 2, s_hat[0][0] = 5 -> product
A_hat[0][1][0] = 6, s_hat[1][0] = 16 -> product
e_hat[0][0]    = 1
```

All 8 coefficients of t_hat[0]:

| i | A[0][0]xs[0] | A[0][1]xs[1] | +e[0] | t_hat[0][i] |
|---|--------------|--------------|-------|-------------|
| 0 | 2x5=10 | 6x16=11 | 1 | (10+11+1) mod 17 = **5** |
| 1 | 11x3=16 | 6x14=16 | 1 | (16+16+1) mod 17 = **16** |
| 2 | 8x0=0 | 9x1=9 | 1 | (0+9+1) mod 17 = **10** |
| 3 | 1x9=9 | 9x15=16 | 1 | (9+16+1) mod 17 = **9** |
| 4 | 15x14=6 | 12x10=1 | 1 | (6+1+1) mod 17 = **8** |
| 5 | 3x16=14 | 12x11=13 | 1 | (14+13+1) mod 17 = **11** |
| 6 | 3x2=6 | 15x8=1 | 1 | (6+1+1) mod 17 = **8** |
| 7 | 6x10=9 | 15x10=14 | 1 | (9+14+1) mod 17 = **7** |

`t_hat[0] = [5, 16, 10, 9, 8, 11, 8, 7]`   `t_hat[1] = [7, 2, 10, 6, 15, 13, 13, 6]`

### Exercise 5: Implement `keygen_t`


In [ ]:
def pointwise_mul(a: list, b: list, q: int) -> list:
    """Pointwise multiply two polynomials mod q."""
    return [x * y % q for x, y in zip(a, b)]

def pointwise_add(a: list, b: list, q: int) -> list:
    """Pointwise add two polynomials mod q."""
    return [(x + y) % q for x, y in zip(a, b)]

def keygen_t(A_hat, s_ntt, e_ntt, q):
    """
    Compute t̂ = Â·ŝ + ê in NTT domain (pointwise operations).
    A_hat:  k×k matrix of NTT-domain polynomials
    s_ntt:  k-vector of NTT-domain polynomials
    e_ntt:  k-vector of NTT-domain polynomials
    Returns: t̂ as a k-vector of NTT-domain polynomials
    """
    k = len(A_hat)
    n = len(A_hat[0][0])
    t_hat = []
    for row in range(k):
        acc = [0] * n                              # accumulator for this row
        for col in range(k):
            prod = pointwise_mul(A_hat[row][col], s_ntt[col], q)
            acc  = pointwise_add(acc, prod, q)
        t_hat.append(pointwise_add(acc, e_ntt[row], q))
    return t_hat

# Detailed trace for t̂[0]
print("=== Building t̂[0] step by step ===\n")
p00 = pointwise_mul(A[0][0], S0_NTT, Q)
p01 = pointwise_mul(A[0][1], S1_NTT, Q)
print(f"Â[0][0] ⊙ ŝ[0] = {p00}")
print(f"Â[0][1] ⊙ ŝ[1] = {p01}")
s00 = pointwise_add(p00, p01, Q)
t0  = pointwise_add(s00, E0_NTT, Q)
print(f"sum              = {s00}")
print(f"+ ê[0]           = {t0}")
print(f"Expected t̂[0]:   {T0}")
assert t0 == T0

print("\n=== Building t̂[1] ===\n")
t1 = pointwise_add(
    pointwise_add(pointwise_mul(A[1][0], S0_NTT, Q),
                  pointwise_mul(A[1][1], S1_NTT, Q), Q),
    E1_NTT, Q)
print(f"t̂[1] = {t1}  expected {T1}")
assert t1 == T1

t_hat = keygen_t(A, [S0_NTT, S1_NTT], [E0_NTT, E1_NTT], Q)
assert t_hat[0] == T0 and t_hat[1] == T1
print("\n✓ Public key t̂ matches the global constants.")
print(f"\npk = (t̂, ρ):  t̂[0] = {t_hat[0]}")
print(f"               t̂[1] = {t_hat[1]}")
print(f"sk = ŝ:         ŝ[0] = {S0_NTT}")
print(f"                ŝ[1] = {S1_NTT}")

## Module 6: Encapsulation

### 6.1 Overview

Bob holds the public key `pk = (t̂, ρ)`. He chooses a message `m` and fresh noise `r, e1, e2`,
and produces ciphertext `(u, v)` that only Alice (with secret key ŝ) can decrypt.

```
u = INTT(Â^T · r̂) + e1
v = INTT(t̂^T · r̂) + e2 + m̂
```

**Transpose:** `Â^T[i][j] = Â[j][i]`, where rows and columns of A are swapped.

**Message encoding:** each bit of m is scaled to the midpoint of Z_q:
```
bit 0 → 0 (near zero)
bit 1 → ⌊q/2⌋ = 8 (near the modulus midpoint)
```
This maximises the distance from a wrong decoding.

### 6.2 Why the transpose?

The algebraic cancellation in decapsulation requires:

```
v − ŝ^T · u = m̂ + small noise terms
```

For this to work, Bob must multiply by `Â^T` (not `Â`) when computing `u`.
This way the `A·s` term inside `t` cancels with the `A^T·r` term inside `u`.

### 6.3 Computing u, full trace for u[0]

```
Â^T row 0 = [Â[0][0], Â[1][0]] (column 0 of A, not row 0)

Â[0][0] ⊙ r̂[0]: [30→13, 55→4, 48→14, 13, 75→7, 30→13, 36→2, 60→9]
NTT-domain, reduced   = [13, 4, 14, 13, 7, 13, 2, 9]

Â[1][0] ⊙ r̂[1]: [98→13, 84→16, 9, 60→9, 40→6, 36→2, 10, 8]
NTT-domain, reduced   = [13, 16, 9, 9, 6, 2, 10, 8]

NTT sum: [9, 3, 6, 5, 13, 15, 12, 0]

INTT([9, 3, 6, 5, 13, 15, 12, 0]) = [10, 8, 14, 3, 0, 4, 3, 7]

+ e1[0] = [0, 0, 0, 1, 0, 0, 0, 0]: u[0] = [10, 8, 14, 4, 0, 4, 3, 7] ✓
```

### Exercise 6: Implement `encapsulate`

In [ ]:
def encode_message(msg: list, q: int) -> list:
    """Encode each bit: 0 → 0,  1 → ⌊q/2⌋."""
    return [bit * (q // 2) for bit in msg]

def matvec_ntt(M, v_ntt, q):
    """Matrix-vector multiply in NTT domain (pointwise per coefficient)."""
    k = len(M); n = len(M[0][0])
    result = []
    for row in range(k):
        acc = [0] * n
        for col in range(k):
            for i in range(n):
                acc[i] = (acc[i] + M[row][col][i] * v_ntt[col][i]) % q
        result.append(acc)
    return result

def encapsulate(A_hat, t_hat, r, e1, e2, msg, roots, inv_roots, q, k, n_inv):
    """
    Kyber encapsulation.
    Returns (u_list, v) where u_list is a k-vector of normal-domain polynomials.
    """
    n = len(r[0])

    # NTT of r polynomials
    r_ntt = [ntt(rr, roots, q) for rr in r]

    # Transpose A: A_T[i][j] = A_hat[j][i]
    A_T = [[A_hat[j][i] for j in range(k)] for i in range(k)]

    # u = INTT(Â^T · r̂) + e1
    ATr_ntt = matvec_ntt(A_T, r_ntt, q)
    u = []
    for row in range(k):
        u_row = intt(ATr_ntt[row], inv_roots, q, n_inv)
        u.append([(u_row[i] + e1[row][i]) % q for i in range(n)])

    # v = INTT(t̂^T · r̂) + e2 + m̂
    tTr_ntt = [sum(t_hat[col][i] * r_ntt[col][i] for col in range(k)) % q
               for i in range(n)]
    v_pre = intt(tTr_ntt, inv_roots, q, n_inv)
    m_enc = encode_message(msg, q)
    v = [(v_pre[i] + e2[i] + m_enc[i]) % q for i in range(n)]

    return u, v

print("=== Encapsulation ===\n")
print(f"Message m     = {MSG}")
print(f"Encoded m̂    = {M_HAT}  (bit→8 if 1, bit→0 if 0)\n")

u_result, v_result = encapsulate(
    A, [T0, T1], [R0, R1], [E1_0, E1_1], E2, MSG,
    ROOTS, INV_ROOTS, Q, K, N_INV
)

print(f"u[0] = {u_result[0]}  expected {U0}")
print(f"u[1] = {u_result[1]}  expected {U1}")
print(f"v    = {v_result}  expected {V}")

assert u_result[0]==U0 and u_result[1]==U1 and v_result==V
print("\n✓ Ciphertext (u, v) matches the global constants.")

## Module 7: Decapsulation

### 7.1 Alice's computation

Alice holds the secret key `ŝ` and receives ciphertext `(u, v)`:

```
w = v − INTT(ŝ^T · NTT(u))
```

Then each coefficient of w is decoded to a bit by **circular distance**:
- `dist_to_0 = min(w[i], q − w[i])`
- `dist_to_q/2 = min(|w[i]−8|, q − |w[i]−8|)`
- **bit = 1** if `dist_to_q/2 < dist_to_0`, else **bit = 0**

### 7.2 Why the algebra works: the cancellation

Expanding v and u algebraically:
```
v = INTT(t̂^T·r̂) + e2 + m̂
 = INTT((Â·ŝ + ê)^T·r̂) + e2 + m̂
 = INTT(ŝ^T·Â^T·r̂) + INTT(ê^T·r̂) + e2 + m̂

ŝ^T·u = INTT(ŝ^T·(Â^T·r̂ + NTT(e1)))
 = INTT(ŝ^T·Â^T·r̂) + INTT(ŝ^T·NTT(e1))

w = v − ŝ^T·u
 = m̂ + INTT(ê^T·r̂) + e2 − INTT(ŝ^T·NTT(e1))
 └──────────────────────────────────────────┘
 NOISE TERMS ONLY
```

The `INTT(ŝ^T·Â^T·r̂)` terms cancel exactly.
Only noise remains, and since s, e, r, e1, e2 all have small coefficients, the noise is small.

### 7.3 Noise budget

```
i w[i] m̂[i] noise = w[i]−m̂[i] |noise| < q/4=4.25?
0 8 8 +0 0 < 4.25 ✓
1 1 0 +1 1 < 4.25 ✓
2 10 8 +2 2 < 4.25 ✓
3 0 0 +0 0 < 4.25 ✓
4 9 8 +1 1 < 4.25 ✓
5 1 0 +1 1 < 4.25 ✓
6 7 8 −1 (7−17=−1) 1 < 4.25 ✓
7 0 0 +0 0 < 4.25 ✓
Max noise = 2. All within budget. Decryption succeeds.
```

### Exercise 7: Implement `decapsulate`

In [ ]:
def decode_bit(coeff: int, q: int) -> int:
    """
    Decode one coefficient to a bit by circular distance.
    Closer to 0     → bit 0
    Closer to q//2  → bit 1
    """
    d0  = min(coeff,          q - coeff)                   # dist to 0
    dq2 = min(abs(coeff-q//2), q - abs(coeff-q//2))        # dist to q/2
    return 1 if dq2 < d0 else 0

def decapsulate(s_hat, u, v, roots, inv_roots, q, k, n_inv):
    """
    Kyber decapsulation.
    Returns the decoded bit list.
    """
    n = len(u[0])

    # Transform u to NTT domain
    u_ntt = [ntt(uu, roots, q) for uu in u]

    # ŝ^T · û (inner product in NTT domain)
    stu_ntt = [
        sum(s_hat[j][i] * u_ntt[j][i] for j in range(k)) % q
        for i in range(n)
    ]

    # INTT back to normal domain
    stu_n = intt(stu_ntt, inv_roots, q, n_inv)

    # w = v − ŝ^T·u
    w = [(v[i] - stu_n[i]) % q for i in range(n)]

    # Decode each coefficient to a bit
    return [decode_bit(c, q) for c in w]

print("=== Decapsulation ===\n")
u0_ntt = ntt(U0, ROOTS, Q)
u1_ntt = ntt(U1, ROOTS, Q)
print(f"NTT(u[0]) = {u0_ntt}  expected {U0_NTT}")
print(f"NTT(u[1]) = {u1_ntt}  expected {U1_NTT}")
assert u0_ntt==U0_NTT and u1_ntt==U1_NTT

stu_ntt = [(S0_NTT[i]*u0_ntt[i] + S1_NTT[i]*u1_ntt[i]) % Q for i in range(N)]
stu_n   = intt(stu_ntt, INV_ROOTS, Q, N_INV)
print(f"\nINTT(ŝ^T·û) = {stu_n}  expected {STU_N}")
assert stu_n == STU_N

w = [(V[i] - stu_n[i]) % Q for i in range(N)]
print(f"\nw = v − ŝ^T·u = {w}  expected {W}")
assert w == W

print("\n=== Noise Budget ===")
print(f"{'i':>3} {'w[i]':>6} {'m̂[i]':>6} {'noise':>8} {'|noise|':>9} {'<q/4?':>8} {'bit':>5}")
decoded = []
for i in range(N):
    noise = (w[i] - M_HAT[i]) % Q
    if noise > Q // 2: noise -= Q
    ok  = "✓" if abs(noise) < Q/4 else "✗ FAIL"
    bit = decode_bit(w[i], Q)
    print(f"{i:>3} {w[i]:>6} {M_HAT[i]:>6} {noise:>+8} {abs(noise):>9} {Q/4:>7.2f} {ok} {bit:>5}")
    decoded.append(bit)

assert decoded == DECODED
print(f"\nDecoded:  {decoded}")
print(f"Original: {MSG}")
print("\n✓ Decapsulation successful: message recovered correctly.")

## Module 8: Full System Test

### 8.1 The complete KeyGen -> Encapsulate -> Decapsulate pipeline

```
KeyGen:       s_hat, e_hat = NTT(s), NTT(e);  t_hat = A * s_hat + e_hat
Encapsulate:  r_hat = NTT(r);  u = INTT(A^T * r_hat) + e1;  v = INTT(t^T * r_hat) + e2 + encode(m)
Decapsulate:  w = v - INTT(s^T * NTT(u));  m' = decode(w);  assert m' == m
```

### 8.2 Why Kyber is post-quantum secure (stated carefully)

Shor's algorithm breaks RSA and ECDSA because those reduce to a **hidden-subgroup /
period-finding** problem: `f(x) = a^x mod N` is periodic, and the Quantum Fourier
Transform extracts the period efficiently.

Module-LWE is **not a period-finding problem at all**. There is no hidden period for the
QFT to lock onto, so Shor's approach has nothing to act on. This is a property of the
problem's structure, not merely of the noise: it is wrong to say "the noise destroys a
period the QFT would exploit," because there is no period to begin with. The small noise
`e` is what makes recovering `s` hard classically and quantumly, but the reason Shor in
particular fails is the absence of the algebraic (periodic) structure he relies on.

```
RSA: f(x) = 7^x mod 15 -> [1,7,4,13,1,7,4,13,...]  period 4  (QFT finds it)
LWE: t = A.s + e  ->  not a periodic function; QFT has no period to find
```

### 8.3 What this toy omits: the FO transform

This notebook implements the **IND-CPA core** (KeyGen / Encrypt / Decrypt). Real ML-KEM
wraps it in the **Fujisaki-Okamoto transform** to reach IND-CCA2: Encaps derives the
shared secret and the encryption coins together from `G(m || H(ek))`, and Decaps
**re-encrypts** and compares, returning a pseudo-random `J(z || c)` on mismatch
(**implicit rejection**). Those steps, omitted here for clarity, are what make a deployed
KEM safe against chosen-ciphertext attacks.

### 8.4 Security levels (approximate)

Rough core-SVP hardness estimates (model-dependent, not exact operation counts):
Kyber-512 ~ Category 1, Kyber-768 ~ Category 3, Kyber-1024 ~ Category 5.

### Exercise 8: Run the full pipeline and verify end-to-end


In [ ]:
print("╔══════════════════════════════════════════════════════════╗")
print("║   CRYSTALS-Kyber  Full System Test                       ║")
print(f"║   q={Q}  n={N}  k={K}  ETA={ETA}                                    ║")
print("╚══════════════════════════════════════════════════════════╝\n")

# KeyGen
print("KeyGen")
s_ntt_kg = [ntt(S0, ROOTS, Q), ntt(S1, ROOTS, Q)]
e_ntt_kg = [ntt(E0, ROOTS, Q), ntt(E1, ROOTS, Q)]
t_hat_kg = keygen_t(A, s_ntt_kg, e_ntt_kg, Q)
assert t_hat_kg[0]==T0 and t_hat_kg[1]==T1
print(f"  t̂[0] = {t_hat_kg[0]}")
print(f"  t̂[1] = {t_hat_kg[1]}")
print("  ✓ KeyGen OK\n")

# Encapsulation
print("Encapsulation")
print(f"  message m = {MSG}")
u_enc, v_enc = encapsulate(
    A, t_hat_kg, [R0,R1], [E1_0,E1_1], E2, MSG,
    ROOTS, INV_ROOTS, Q, K, N_INV
)
assert u_enc[0]==U0 and u_enc[1]==U1 and v_enc==V
print(f"  u[0] = {u_enc[0]}")
print(f"  u[1] = {u_enc[1]}")
print(f"  v    = {v_enc}")
print("  ✓ Encapsulation OK\n")

# Decapsulation
print("Decapsulation")
m_recovered = decapsulate(s_ntt_kg, u_enc, v_enc, ROOTS, INV_ROOTS, Q, K, N_INV)
assert m_recovered == MSG
print(f"  w       = {W}")
print(f"  decoded = {m_recovered}")
print(f"  original= {MSG}")
print("  ✓ Decapsulation OK\n")

# Noise report
print("Noise Budget")
for i in range(N):
    noise = (W[i] - M_HAT[i]) % Q
    if noise > Q//2: noise -= Q
    bar = "█" * abs(noise) + "░" * (Q//4 - abs(noise))
    print(f"  coeff[{i}]  w={W[i]:2d}  m̂={M_HAT[i]}  noise={noise:+2d}  [{bar}]  < q/4={Q//4}")

print("\n╔══════════════════════════════════════════════════════════╗")
print("║    FULL SYSTEM TEST PASSED                              ║")
print("║  KeyGen → Encapsulate → Decapsulate  ✓                   ║")
print("╚══════════════════════════════════════════════════════════╝")